In [ ]:
from pyspark.sql.functions import (
    col, trim, when, max as spark_max, explode, lit, from_json
)

from delta.tables import DeltaTable

tabela_nome = "silver_clientes"

if spark.catalog.tableExists("controle_pipeline"):
        df_controle = spark.table("controle_pipeline").filter(col("tabela") == tabela_nome)

        if df_controle.count() > 0: 
            ultimo_processamento = df_controle.select("ultimo_ts").collect()[0][0]
        else:
            ultimo_processamento = None
else: 
    ultimo_processamento = None

In [ ]:
nome_tabela = "clientes"
caminho_bronze = f"Files/bronze/{nome_tabela}"
df_bronze = spark.read.format("delta").load(caminho_bronze)

if ultimo_processamento:
    df_bronze = df_bronze.filter(
        col("ingestion_ts") > ultimo_processamento
    )

if df_bronze.rdd.isEmpty():
    print("Sem dados novos para processar.")
    dbutils.notebook.exit("No data")

In [ ]:
from pyspark.sql.types import *

schema = StructType([
    StructField("bairro", StringType()),
    StructField("bloquear_faturamento", StringType()),
    StructField("cep", StringType()),
    StructField("cidade", StringType()),
    StructField("cidade_ibge", StringType()),
    StructField("cnae", StringType()),
    StructField("cnpj_cpf", StringType()),
    StructField("codigo_cliente_integracao", StringType()),
    StructField("codigo_cliente_omie", LongType()),
    StructField("codigo_pais", StringType()),
    StructField("complemento", StringType()),

    StructField("dadosBancarios", StructType([
            StructField("agencia", StringType()),
            StructField("cChavePix", StringType()),
            StructField("codigo_banco", StringType()),
            StructField("conta_corrente", StringType()),
            StructField("doc_titular", StringType()),
            StructField("nome_titular", StringType()),
            StructField("transf_padrao", StringType()),   
    ])),

    StructField("email", StringType()),
    StructField("endereco", StringType()),
    StructField("endereco_numero", StringType()),
    StructField("enviar_anexos", StringType()),
    StructField("estado", StringType()),
    StructField("exterior", StringType()),
    StructField("inativo", StringType()),

    StructField("info", StructType([
        StructField("cImpAPI", StringType()),
        StructField("dAlt", StringType()),
        StructField("dInc", StringType()),
        StructField("hAlt", StringType()),
        StructField("hInc", StringType()),
        StructField("uAlt", StringType()),
        StructField("uInc", StringType()),
    ])),

    StructField("inscricao_estadual", StringType()),
    StructField("inscricao_municipal", StringType()),
    StructField("nome_fantasia", StringType()),
    StructField("pessoa_fisica", StringType()),
    StructField("razao_social", StringType()),

    StructField("recomendacoes", StructType([
        StructField("gerar_boletos", StringType()),
        StructField("tipo_assinante", StringType()),
    ])),
    
    StructField("tags", ArrayType(
        StructType([
            StructField("tag", StringType())
        ])
    )),

    StructField("telefone1_ddd", StringType()),
    StructField("telefone1_numero", StringType()),
    StructField("tipo_atividade", StringType()),
    StructField("empresa", StringType())

])

In [ ]:

df_bronze = df_bronze.withColumn("json", from_json(col("raw_json"), schema))
df_silver = df_bronze.select("json.*", "ingestion_ts")

In [ ]:

for c, t in df_silver.dtypes:
    if t == "string":
        df_silver = df_silver.withColumn(
            c,
            when(trim(col(c)) == "", None)
            .otherwise(trim(col(c)))
        )

df_silver = df_silver \
    .withColumn("agencia", col("dadosBancarios.agencia")) \
    .withColumn("chave_pix", col("dadosBancarios.cChavePix")) \
    .withColumn("codigo_banco", col("dadosBancarios.codigo_banco")) \
    .withColumn("conta_corrente", col("dadosBancarios.conta_corrente")) \
    .withColumn("doc_titular", col("dadosBancarios.doc_titular")) \
    .withColumn("nome_titular", col("dadosBancarios.nome_titular")) \
    .withColumn("transf_padrao", col("dadosBancarios.transf_padrao")) \
    .withColumn("data_inclusao", col("info.dInc")) \
    .withColumn("data_alteracao", col("info.dAlt")) \
    .withColumn("usuario_inclusao", col("info.uInc")) \
    .withColumn("usuario_alteracao", col("info.uAlt"))

df_silver = df_silver.drop("dadosBancarios", "info", "recomendacoes", "tags")

df_silver = df_silver.withColumn(
    "inativo_flag",
    when(col("inativo") == "S", True).otherwise(False)
)

df_silver = df_silver.dropDuplicates(["codigo_cliente_omie", "empresa"])


In [ ]:

if spark.catalog.tableExists("silver_clientes"):

    delta_table = DeltaTable.forName(spark, "silver_clientes")

    (delta_table.alias("t")
            .merge(
                df_silver.alias("s"),
                """
                t.codigo_cliente_omie = s.codigo_cliente_omie
                AND t.empresa = s.empresa
                """
            )
            .whenMatchedUpdateAll()
            .whenNotMatchedInsertAll()
            .execute()  
    )

else: 
    df_silver.write.format("delta").mode("overwrite").saveAsTable("silver_clientes")

In [ ]:
novo_max_ts = df_bronze.agg(
    spark_max("ingestion_ts").alias("max_ts")).collect()[0]["max_ts"]

df_update = spark.createDataFrame(
    [(tabela_nome, novo_max_ts)],
    ["tabela", "ultimo_ts"]
)

if spark.catalog.tableExists("controle_pipeline"):

    delta_ctrl = DeltaTable.forName(spark, "controle_pipeline")

    (delta_ctrl.alias("t")
        .merge(
            df_update.alias("s"),
            "t.tabela = s.tabela"
        )
        .whenMatchedUpdateAll()
        .whenNotMatchedInsertAll()
        .execute()
    )

else: 
    df_update.write.format("delta").mode("overwrite").saveAsTable("controle_pipeline")